# ZARX-1B Training Notebook (Kaggle)
1B parameter code + ARC reasoning model trained from scratch.

**GPU:** T4/P100 (16GB VRAM) | **Session:** 9 hours max

## Persistence (NEVER blocks the process!)
- GitHub: pushes small files to codedbytahir/zarx-checkpoints (<95MB)
- HuggingFace: pushes everything to Chvigo/zarx-checks (no size limit)
- Kaggle Output: saves final checkpoint

## Setup
1. Create a new notebook on Kaggle
2. Settings -> Accelerator -> GPU T4 x2 or P100
3. Settings -> Internet -> ON (required!)
4. Run cells in order
5. Training auto-resumes from checkpoints on GitHub/HF

In [ ]:
#@title 1. Install Dependencies
!pip install -q torch>=2.1.0 bitsandbytes wandb huggingface_hub
!pip install -q datasets tokenizers accelerate datasketch
import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name()}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'BF16: {torch.cuda.is_bf16_supported()}')
if torch.cuda.is_available() and ('A100' in torch.cuda.get_device_name() or 'L4' in torch.cuda.get_device_name()):
    !pip install -q flash-attn --no-build-isolation
print('Dependencies installed!')

In [ ]:
#@title 2. Set Credentials
import os

# --- FILL THESE IN ---
HF_TOKEN = '' #@param {type:"string"}
WANDB_KEY = '' #@param {type:"string"}
GH_TOKEN = '' #@param {type:"string"}

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print('HF login OK!')

if WANDB_KEY:
    import wandb
    wandb.login(key=WANDB_KEY)
    print('W&B login OK!')

print('Credentials set!')

In [ ]:
#@title 3. Clone ZARX-1B + Checkpoint Repo
%cd /kaggle/working
!rm -rf ZARX-1B
!git clone https://github.com/codedbytahir/ZARX-1B.git
%cd ZARX-1B

# Clone checkpoint repo
if GH_TOKEN:
    !rm -rf /kaggle/working/zarx-checkpoints
    !git clone https://{GH_TOKEN}@github.com/codedbytahir/zarx-checkpoints.git /kaggle/working/zarx-checkpoints

print('Repos cloned!')

In [ ]:
#@title 4A. Option A: Full Day 1 Setup (first time on Kaggle)
# Re-downloads all data, processes it, trains tokenizer, test training.
# Saves everything to GitHub + HuggingFace (no Drive on Kaggle).
# Takes ~3-4 hours total.

!python scripts/setup_colab_day1.py \
    --hf_token "{HF_TOKEN}" \
    --wandb_key "{WANDB_KEY}" \
    --hf_repo "Chvigo/zarx-checks" \
    --github_token "{GH_TOKEN}" \
    --github_repo "codedbytahir/zarx-checkpoints"

In [ ]:
#@title 4B. Option B: Resume from HuggingFace (if data was saved there)
# Restores data + tokenizer + checkpoints from HuggingFace.
# Then starts training from where you left off.

import shutil, os
from pathlib import Path

DATA = Path('/kaggle/working/data')
DATA.mkdir(parents=True, exist_ok=True)

# Try restoring from HuggingFace
if HF_TOKEN:
    from huggingface_hub import HfApi, snapshot_download, hf_hub_download
    api = HfApi(token=HF_TOKEN)
    
    # Restore processed data
    try:
        print('Restoring processed data from HuggingFace...')
        snapshot_download(
            repo_id='Chvigo/zarx-checks',
            allow_patterns=['data/processed/*'],
            repo_type='model',
            token=HF_TOKEN,
            local_dir=str(DATA.parent),
        )
        print('Processed data restored!')
    except Exception as e:
        print(f'No processed data on HF: {e}')
    
    # Restore tokenizer
    try:
        print('Restoring tokenizer from HuggingFace...')
        hf_hub_download(
            repo_id='Chvigo/zarx-checks',
            filename='data/tokenizer/tokenizer.json',
            repo_type='model',
            token=HF_TOKEN,
            local_dir='/kaggle/working/ZARX-1B',
        )
        # Move to configs if needed
        tok_path = Path('/kaggle/working/ZARX-1B/data/tokenizer/tokenizer.json')
        if tok_path.exists():
            shutil.copy2(str(tok_path), '/kaggle/working/ZARX-1B/configs/tokenizer.json')
        print('Tokenizer restored!')
    except Exception as e:
        print(f'No tokenizer on HF: {e}')
    
    # Restore latest checkpoint
    try:
        print('Restoring checkpoint from HuggingFace...')
        hf_hub_download(
            repo_id='Chvigo/zarx-checks',
            filename='checkpoint-latest.pt',
            repo_type='model',
            token=HF_TOKEN,
            local_dir='/kaggle/working/checkpoints',
        )
        print('Checkpoint restored! Training will auto-resume.')
    except Exception as e:
        print(f'No checkpoint on HF: {e}')
else:
    print('No HF token set. Cannot restore from HuggingFace.')

os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
print('Restore complete!')

In [ ]:
#@title 5. Start Training (auto-resumes, saves to GitHub + HF)
!python /kaggle/working/ZARX-1B/src/train.py \
    --model_config /kaggle/working/ZARX-1B/configs/model_config.json \
    --tokenizer_path /kaggle/working/ZARX-1B/configs/tokenizer.json \
    --data_path /kaggle/working/data/processed \
    --micro_batch_size 1 \
    --gradient_accumulation_steps 32 \
    --learning_rate 3e-4 \
    --warmup_steps 2000 \
    --total_tokens 10000000000 \
    --use_8bit_adam \
    --checkpoint_dir /kaggle/working/checkpoints \
    --drive_dir /kaggle/working/zarx-persist \
    --github_token "{GH_TOKEN}" \
    --github_repo codedbytahir/zarx-checkpoints \
    --hf_repo_id Chvigo/zarx-checks \
    --hf_token "{HF_TOKEN}" \
    --save_every_local 100 \
    --save_every_drive 999999 \
    --save_every_github 1000 \
    --save_every_hf 1000 \
    --log_every_steps 10 \
    --wandb_project zarx-1b \
    --output_dir /kaggle/working/zarx-1b-final

In [ ]:
#@title 6. EMERGENCY SAVE (run if session is about to die!)
import shutil
from pathlib import Path

# Save checkpoint to Kaggle output
if Path('/kaggle/working/checkpoints/checkpoint-latest.pt').exists():
    shutil.copy('/kaggle/working/checkpoints/checkpoint-latest.pt', '/kaggle/working/zarx-final-checkpoint.pt')
    print('Checkpoint saved to Kaggle output!')

# Push to GitHub
if GH_TOKEN and Path('/kaggle/working/zarx-checkpoints').exists():
    try:
        # Copy metadata (small files)
        if Path('/kaggle/working/checkpoints/training_metadata.json').exists():
            shutil.copy2('/kaggle/working/checkpoints/training_metadata.json', '/kaggle/working/zarx-checkpoints/')
        !cd /kaggle/working/zarx-checkpoints && git add -A && git commit -m 'kaggle emergency save' && git push https://{GH_TOKEN}@github.com/codedbytahir/zarx-checkpoints.git main
        print('Pushed to GitHub!')
    except Exception as e:
        print(f'GitHub push failed: {e}')

# Push to HuggingFace (saves large checkpoints!)
if HF_TOKEN:
    try:
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        if Path('/kaggle/working/checkpoints/checkpoint-latest.pt').exists():
            api.upload_file(
                path_or_fileobj='/kaggle/working/checkpoints/checkpoint-latest.pt',
                path_in_repo='checkpoint-latest.pt',
                repo_id='Chvigo/zarx-checks',
                repo_type='model',
            )
        if Path('/kaggle/working/checkpoints/training_metadata.json').exists():
            api.upload_file(
                path_or_fileobj='/kaggle/working/checkpoints/training_metadata.json',
                path_in_repo='training_metadata.json',
                repo_id='Chvigo/zarx-checks',
                repo_type='model',
            )
        print('Pushed to HuggingFace!')
    except Exception as e:
        print(f'HF push failed: {e}')

print('EMERGENCY SAVE COMPLETE!')